# onnx_export

> Export a :class:`~segmentation_tools_pt.model_wrapper.TiledInferenceModelONNX`
> to ONNX format.
>
> Because `TiledInferenceModelONNX` bakes tile positions statically at
> construction time, the exported graph is fully traced and compatible
> with all ONNX runtimes.  The default opset is **12** for maximum
> edge / embedded runtime compatibility.  Dynamic batch axes are
> disabled by default (single-image inference).
>
> **Optional dependency:** `onnxsim` (onnx-simplifier) is used when
> `OnnxExportConfig.simplify = True`.  Install it with
> `uv add onnxsim` or leave simplification off.

In [2]:
#| default_exp onnx_export

In [3]:
#| hide
from nbdev.showdoc import *

In [4]:
#| export
from __future__ import annotations

import logging
import os
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Union

import numpy as np
import torch
import torch.nn as nn
from torch import Tensor

logger = logging.getLogger(__name__)

## Export configuration

In [5]:
#| export
@dataclass
class OnnxExportConfig:
    """Configuration for ONNX export.
    Example::

        cfg = OnnxExportConfig(
            opset_version=12,
            input_names=["image"],
            output_names=["segmentation"],
            simplify=True,
        )
    """
    opset_version:    int              = 12
    input_names:      List[str]        = field(default_factory=lambda: ["image"])# input name
    output_names:     List[str]        = field(default_factory=lambda: ["segmentation"])# output name
    dynamic_axes:     Dict             = field(default_factory=dict) # dynamic axes
    simplify:         bool             = False # simplify the graph
    simplify_kwargs:  Dict             = field(default_factory=dict) # simplify kwargs
    verbose:          bool             = False # verbose
    dynamo:           bool             = False # False = legacy TorchScript tracer (avoids "expand 2->1" on tiled models)

## Export function

In [6]:
#| export
def export_to_onnx(
    model:        nn.Module,
    sample_input: Tensor,
    save_path:    str | Path,
    cfg:          Optional[OnnxExportConfig] = None,
) -> Path:
    """Export ``model`` to an ONNX file.

    The function:

    1. Sets the model to eval mode.
    2. Calls ``torch.onnx.export`` with the provided configuration.
    3. Optionally runs ``onnxsim.simplify`` for graph-level optimisations.
    4. Validates the exported graph with ``onnx.checker.check_model``.

    Args:
        model:        Any ``nn.Module`` — typically a
                      :class:`~segmentation_tools_pt.model_wrapper.TiledInferenceModel`.
        sample_input: Representative input tensor used for tracing.
                      Shape must match the model's expected input.
        save_path:    Destination ``.onnx`` file path.
        cfg:          Export configuration.  Defaults to
                      :class:`OnnxExportConfig` with all defaults.

    Returns:
        Resolved path to the saved ONNX file.

    Raises:
        ImportError: If ``onnx`` is not installed.
        ImportError: If ``cfg.simplify=True`` and ``onnxsim`` is not installed.
        onnx.checker.ValidationError: If the exported graph fails ONNX validation.

    Example::

        from segmentation_tools_pt.onnx_export import export_to_onnx, OnnxExportConfig
        from segmentation_tools_pt.model_wrapper import TiledInferenceModel

        cfg = OnnxExportConfig(simplify=True)
        export_to_onnx(tiled_model, sample_input=torch.randn(1,3,1152,1632),
                       save_path="tiled_seg.onnx", cfg=cfg)
    """
    try:
        import onnx
    except ImportError as exc:
        raise ImportError(
            "onnx is required for export.  Install it with: uv add onnx"
        ) from exc

    cfg       = cfg or OnnxExportConfig()
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    model.eval()

    logger.info(
        "Exporting ONNX: opset=%d  input=%s  → %s",
        cfg.opset_version, tuple(sample_input.shape), save_path,
    )

    with torch.no_grad():
        torch.onnx.export(
            model,
            sample_input,
            str(save_path),
            input_names=cfg.input_names,
            output_names=cfg.output_names,
            dynamic_axes=cfg.dynamic_axes if cfg.dynamic_axes else None,
            opset_version=cfg.opset_version,
            verbose=cfg.verbose,
            dynamo=cfg.dynamo,
        )

    # --- Optional graph simplification ---
    if cfg.simplify:
        try:
            import onnxsim
        except ImportError as exc:
            raise ImportError(
                "onnxsim is required for simplification.  Install it with: uv add onnxsim"
            ) from exc

        logger.info("Running onnxsim graph simplification ...")
        onnx_model = onnx.load(str(save_path))
        simplified, check_ok = onnxsim.simplify(onnx_model, **cfg.simplify_kwargs)
        if not check_ok:
            logger.warning("onnxsim could not fully simplify the graph — saving original.")
        else:
            onnx.save(simplified, str(save_path))
            logger.info("Simplified ONNX graph saved.")

    # --- Structural validation ---
    onnx_model = onnx.load(str(save_path))
    onnx.checker.check_model(onnx_model)
    logger.info("ONNX validation passed: %s", save_path)

    return save_path

## ONNX Runtime validation helper

In [7]:
#| export
def validate_with_onnxruntime(
    onnx_path:    str | Path, # Path to the exported ``.onnx``
    sample_input: Tensor, # The **same** tensor used for export.
    input_name:   str   = "image", # ONNX graph input name (must match ``OnnxExportConfig.input_names[0]``).
    rtol:         float = 1e-3, # Relative tolerance for numeric comparison.
    atol:         float = 1e-5, # Absolute tolerance for numeric comparison.
    pytorch_ref:  Optional[nn.Module] = None, # If provided, compare ORT output against this PyTorch model's output.
) -> bool:
    """Run a forward pass with ONNX Runtime and optionally compare against PyTorch.

    """
    try:
        import onnxruntime as ort
    except ImportError as exc:
        raise ImportError(
            "onnxruntime is required for ORT validation.  Install it with: uv add onnxruntime"
        ) from exc

    # Suppress ORT warnings (provider fallback, serialization, etc.)
    if hasattr(ort, "set_default_logger_severity"):
        ort.set_default_logger_severity(3)  # 3 = ERROR only

    # Configure session for multi-threading
    n_threads = os.cpu_count() or 4
    sess_opts = ort.SessionOptions()
    sess_opts.intra_op_num_threads = n_threads
    sess_opts.inter_op_num_threads = n_threads
    sess_opts.log_severity_level = 3  # 3 = ERROR only, suppresses warnings

    sess = ort.InferenceSession(
        str(onnx_path),
        sess_options=sess_opts,
        providers=["CUDAExecutionProvider", "CPUExecutionProvider"],
    )
    ort_inputs  = {input_name: sample_input.cpu().numpy()}
    ort_outputs = sess.run(None, ort_inputs)
    ort_out     = torch.from_numpy(ort_outputs[0])

    if pytorch_ref is not None:
        pytorch_ref.eval()
        with torch.no_grad():
            pt_out = pytorch_ref(sample_input.cpu())
        max_diff = (ort_out - pt_out).abs().max().item()
        if not torch.allclose(ort_out, pt_out, rtol=rtol, atol=atol):
            raise AssertionError(
                f"ORT vs PyTorch mismatch: max_diff={max_diff:.6f}  (rtol={rtol}, atol={atol})"
            )
        logger.info("ORT↔PyTorch max diff: %.2e — validation PASSED", max_diff)

    return True

## ONNX Inference helpers

Two small, single-responsibility functions:

- `load_onnx_model` — open an `.onnx` file and return a configured
  `onnxruntime.InferenceSession`.  Automatically selects `CUDAExecutionProvider`
  when a GPU is available, otherwise falls back to CPU.
- `run_onnx_inference` — accept a PyTorch `Tensor` (or NumPy array), run it
  through an `InferenceSession`, and return a list of `Tensor` outputs — one
  per graph output node.

In [10]:
#| export
def load_onnx_model(
    onnx_path:   str | Path,                   # Path to the ``.onnx`` file.
    providers:   Optional[List[str]] = None,   # ORT execution providers. ``None`` = auto-detect.
    num_threads: Optional[int]       = None,   # Intra/inter-op thread count. ``None`` = ``os.cpu_count()``.
) -> "ort.InferenceSession":
    """Load an ONNX file and return a ready-to-use ``InferenceSession``.

    Provider priority when ``providers=None``:

    1. ``CUDAExecutionProvider`` — used when ``onnxruntime-gpu`` is installed and
       a CUDA device is available.
    2. ``CPUExecutionProvider`` — always available as a fallback.

    Args:
        onnx_path:   Path to the exported ``.onnx`` file.
        providers:   Explicit provider list, e.g. ``["CPUExecutionProvider"]``.
                     Pass ``None`` to let the function decide.
        num_threads: Number of threads for intra- and inter-op parallelism.
                     Defaults to ``os.cpu_count()``.

    Returns:
        Configured ``onnxruntime.InferenceSession``.

    Raises:
        ImportError:   If ``onnxruntime`` is not installed.
        FileNotFoundError: If ``onnx_path`` does not exist.

    Example::

        session = load_onnx_model("outputs/tiled_seg.onnx")
        print(session.get_inputs()[0].name)   # e.g. "inputs"
    """
    try:
        import onnxruntime as ort
    except ImportError as exc:
        raise ImportError(
            "onnxruntime is required.  Install it with: uv add onnxruntime"
        ) from exc

    onnx_path = Path(onnx_path)
    if not onnx_path.exists():
        raise FileNotFoundError(f"ONNX model not found: {onnx_path}")

    if providers is None:
        available = ort.get_available_providers()
        providers = (
            ["CUDAExecutionProvider", "CPUExecutionProvider"]
            if "CUDAExecutionProvider" in available
            else ["CPUExecutionProvider"]
        )

    # Detect if running in a notebook environment
    def _is_notebook():
        try:
            from IPython import get_ipython
            shell = get_ipython()
            if shell is None:
                return False
            if shell.__class__.__name__ == "ZMQInteractiveShell":
                return True  # Jupyter notebook or qtconsole
            elif shell.__class__.__name__ == "TerminalInteractiveShell":
                return False  # Terminal running IPython
            else:
                return False  # Other types
        except ImportError:
            return False

    if _is_notebook():
        n_threads = 1
    else:
        n_threads = num_threads or (os.cpu_count() or 4)

    sess_opts = ort.SessionOptions()
    sess_opts.intra_op_num_threads = n_threads
    sess_opts.inter_op_num_threads = n_threads
    sess_opts.log_severity_level   = 3  # ERROR only — suppress provider-fallback noise

    session = ort.InferenceSession(
        str(onnx_path),
        sess_options=sess_opts,
        providers=providers,
    )

    active_provider = session.get_providers()[0]
    logger.info("Loaded %s  →  provider: %s", onnx_path.name, active_provider)
    return session

In [11]:
#| export
def run_onnx_inference(
    session:    "ort.InferenceSession",           # Session from :func:`load_onnx_model`.
    x:          Union[Tensor, np.ndarray],        # Input — ``Tensor`` or ``np.ndarray``, shape ``(1, C, H, W)``.
    input_name: Optional[str] = None,            # Override graph input name. ``None`` = read from session.
) -> List[Tensor]:
    """Run a single forward pass through an ONNX model.

    Converts a PyTorch ``Tensor`` (or NumPy array) to ``float32``, runs it
    through the session, and returns every output as a ``torch.Tensor``.

    Args:
        session:    An ``InferenceSession`` returned by :func:`load_onnx_model`.
        x:          Input data. If a ``Tensor``, it is moved to CPU and cast to
                    ``float32`` before passing to ORT.
        input_name: Name of the graph's input node.  When ``None`` the name is
                    read automatically from ``session.get_inputs()[0].name``.

    Returns:
        List of ``Tensor`` outputs, one per graph output node.
        For a single-output model: ``outputs[0]`` is the result.

    Example::

        session = load_onnx_model("outputs/tiled_seg.onnx")
        x = torch.rand(1, 1, 1152, 1632)
        outputs = run_onnx_inference(session, x)
        mask = outputs[0]          # Tensor  (1, 1, 1152, 1632)
        print(mask.shape, mask.dtype)
    """
    # Resolve input name once
    if input_name is None:
        input_name = session.get_inputs()[0].name

    # Normalise input → float32 numpy on CPU
    if isinstance(x, Tensor):
        x_np: np.ndarray = x.detach().cpu().float().numpy()
    else:
        x_np = np.asarray(x, dtype=np.float32)

    raw_outputs = session.run(None, {input_name: x_np})
    return [torch.from_numpy(out) for out in raw_outputs]

In [ ]:
#| eval: false
# --- smoke test: load model exported by the demo below and run inference ---
session = load_onnx_model("outputs/tiled_seg.onnx")

# auto-read input name from the graph
print("Input  nodes:", [i.name for i in session.get_inputs()])
print("Output nodes:", [o.name for o in session.get_outputs()])
print("Provider:    ", session.get_providers()[0])

x_test   = torch.rand(1, IN_C, INPUT_H, INPUT_W)
outputs  = run_onnx_inference(session, x_test)

assert len(outputs) == 1, f"Expected 1 output, got {len(outputs)}"
assert outputs[0].shape == (1, OUT_C, INPUT_H, INPUT_W), f"Wrong shape: {outputs[0].shape}"
print(f"[OK] Inference: {tuple(x_test.shape)} → {tuple(outputs[0].shape)}  dtype={outputs[0].dtype}")

## Export demo

The cell below is tagged `#| eval: false` so it is skipped during `nbdev_test`
and CI runs.  Run it manually after installing `onnx` and (optionally) `onnxruntime`.

In [6]:
#| eval: false
import torch
import torch.nn as nn
from segmentation_tools_pt.model_wrapper import TiledInferenceModelONNX
from segmentation_tools_pt.onnx_export import export_to_onnx, OnnxExportConfig, validate_with_onnxruntime

INPUT_H, INPUT_W = 1152, 1632
IN_C,  OUT_C     = 1, 1
TILE_SIZE, OVERLAP = 256, 32

# ── Build a dummy stand-in for your real model ───────────────────────────────
class MockSegModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(IN_C, OUT_C, kernel_size=1)
    def forward(self, x):
        return self.conv(x)


In [7]:

tiled = TiledInferenceModelONNX(
    model=MockSegModel(),
    tile_size=TILE_SIZE,
    overlap=OVERLAP,
    input_h=INPUT_H,
    input_w=INPUT_W,
    apply_postprocessing=False,
)

sample = torch.randn(1, IN_C, INPUT_H, INPUT_W)
sample.shape

torch.Size([1, 1, 1152, 1632])

In [8]:

export_cfg = OnnxExportConfig(
    opset_version=12,
    input_names=["inputs"],
    output_names=["outputs"],
    simplify=False,   # requires: uv add onnxsim
)
export_cfg

OnnxExportConfig(opset_version=12, input_names=['inputs'], output_names=['outputs'], dynamic_axes={}, simplify=False, simplify_kwargs={}, verbose=False, dynamo=False)

In [9]:
Path("outputs").mkdir(parents=True, exist_ok=True)

In [10]:

saved = export_to_onnx(
    tiled, 
    sample, 
    save_path="outputs/tiled_seg.onnx", 
    cfg=export_cfg
    )
print(f"Saved: {saved}")


/home/ai_dsx.work/data/projects/goni/image_tiling_stiching_pt/.venv/lib/python3.12/site-packages/torch/onnx/_internal/torchscript_exporter/jit_utils.py:305: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at /pytorch/torch/csrc/jit/passes/onnx/constant_fold.cpp:178.)
  _C._jit_pass_onnx_node_shape_type_inference(node, params_dict, opset_version)
/home/ai_dsx.work/data/projects/goni/image_tiling_stiching_pt/.venv/lib/python3.12/site-packages/torch/onnx/_internal/torchscript_exporter/utils.py:714: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at /pytorch/torch/csrc/jit/passes/onnx/constant_fold.cpp:178.)
  _C._jit_pass_onnx_graph_shape_type_inference(
/home/ai_dsx.work/data/projects/goni/image_tiling_stiching_pt/.venv/lib/python3.12/site-packages/torch/onnx/_internal/torchscript_ex

Saved: outputs/tiled_seg.onnx


In [11]:

# ── ORT validation ────────────────────────────────────────────────────────────
ok = validate_with_onnxruntime(
    saved, sample, input_name=export_cfg.input_names[0], pytorch_ref=tiled,
)
print(f"ORT validation: {'PASS' if ok else 'FAIL'}")

ORT validation: PASS


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()